# Wstęp do IPW

**Autor:** Maciej Beręsewicz

## Pakiety

In [1]:
## pip install pandas numpy matplotlib statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

## Dane

Dane wyeksportowane z pakietu `nonprobsvy` (R):

- `admin` -- próba nielosowa $S_A$ (dane z CBOP)
- `jvs` -- próba losowa $S_B$ (dane z badania popyt na pracę, z wagami)

In [ ]:
jvs = pd.read_csv('../data/jvs.csv')
admin = pd.read_csv('../data/admin.csv')
print(f'admin: {admin.shape}, jvs: {jvs.shape}')
admin.head()

In [ ]:
jvs.head()

## Pseudo-likelihood

$$
\ell^*(\boldsymbol{\gamma}) = \sum_{i \in S_{A}} \log \left\{\frac{\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)}{1-\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)}\right\} + \sum_{i \in S_{B}} d_i^B \log \left\{1-\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)\right\}.
$$

$$
\boldsymbol{U}(\boldsymbol{\gamma}) = \sum_{i \in S_A} \boldsymbol{x}_i - \sum_{i \in S_B} d_i^B \pi(\boldsymbol{x}_i, \boldsymbol{\gamma}) \boldsymbol{x}_i.
$$

## Funkcje pomocnicze

In [ ]:
wspol = ['private', 'size', 'nace', 'region']

def fit_ps_model(admin, jvs, formula_vars):
    """Estymacja propensity score z pseudo-likelihood (GLM z wagami JVS)."""
    jvs_sub = jvs[formula_vars].copy()
    jvs_sub['source'] = 0
    jvs_sub['weight'] = jvs['weight'].values
    admin_sub = admin[formula_vars].copy()
    admin_sub['source'] = 1
    admin_sub['weight'] = 1.0
    combined = pd.concat([jvs_sub, admin_sub], ignore_index=True)
    combined_dum = pd.get_dummies(combined[formula_vars],
                                  columns=[v for v in formula_vars if v != 'private'],
                                  dtype=float, drop_first=True)
    X = sm.add_constant(combined_dum)
    y = combined['source']
    model = sm.GLM(y, X, family=sm.families.Binomial(),
                   freq_weights=combined['weight']).fit()
    ps_hat = model.predict(X)
    ps_admin = ps_hat[y == 1].values
    return model, ps_admin

def ipw_estimate(admin, jvs, formula_vars, target_var='single_shift'):
    """Estymator IPW (Hajek) średniej zmiennej celu."""
    model, ps_admin = fit_ps_model(admin, jvs, formula_vars)
    w = 1.0 / ps_admin
    mu_hat = np.average(admin[target_var].values, weights=w)
    return {'mean': mu_hat, 'N_hat': w.sum(), 'model': model,
            'ps_admin': ps_admin, 'ipw_weights': w}

def check_balance(admin, jvs, var, ipw_weights):
    """Balans zmiennej kategorycznej."""
    cats = sorted(admin[var].unique())
    rows = []
    for c in cats:
        rows.append({'Kategoria': c,
                     'CBOP (raw)': round((admin[var] == c).mean(), 4),
                     'CBOP (IPW)': round(np.average(admin[var] == c, weights=ipw_weights), 4),
                     'JVS (ważone)': round(np.average(jvs[var] == c, weights=jvs['weight']), 4)})
    return pd.DataFrame(rows)

## Przykład 1: IPW z jedną zmienną

$$P(R_A = 1 | \text{size})$$

In [ ]:
res1 = ipw_estimate(admin, jvs, ['size'])
print(f'Szacunek IPW (single_shift): {res1["mean"]:.4f}')
print(f'Szacowana wielkość populacji: {res1["N_hat"]:.0f}')

In [ ]:
print('Współczynniki propensity score:')
res1['model'].params.round(4)

In [ ]:
print('Rozkład wag IPW:')
pd.Series(res1['ipw_weights']).describe().round(4)

In [ ]:
check_balance(admin, jvs, 'size', res1['ipw_weights'])

In [ ]:
check_balance(admin, jvs, 'private', res1['ipw_weights'])

## Przykład 2: IPW ze wszystkimi zmiennymi

$$P(R_A = 1 | \text{size, nace, region, private})$$

In [ ]:
res2 = ipw_estimate(admin, jvs, wspol)
print(f'Szacunek IPW (single_shift): {res2["mean"]:.4f}')
print(f'Szacowana wielkość populacji: {res2["N_hat"]:.0f}')

In [ ]:
pd.DataFrame({
    'Model': ['~size', '~size+nace+region+private'],
    'Szacunek': [round(res1['mean'], 4), round(res2['mean'], 4)],
    'N_hat': [round(res1['N_hat'], 0), round(res2['N_hat'], 0)]
})

In [ ]:
res2['model'].params.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(res2['ipw_weights'], bins='fd', color='steelblue', edgecolor='white')
ax.set_xlabel('Waga IPW'); ax.set_ylabel('Liczba firm')
ax.set_title('Rozkład wag IPW (model 2)')
plt.tight_layout(); plt.show()

In [ ]:
print('Balans size -- model 1:')
print(check_balance(admin, jvs, 'size', res1['ipw_weights']))
print('\nBalans size -- model 2:')
print(check_balance(admin, jvs, 'size', res2['ipw_weights']))

## Wykresy Love'a

In [ ]:
def love_plot(admin, jvs, ipw_weights, title=''):
    admin_dum = pd.get_dummies(admin[wspol], columns=['size', 'nace', 'region'], dtype=float)
    jvs_dum = pd.get_dummies(jvs[wspol], columns=['size', 'nace', 'region'], dtype=float)
    all_cols = sorted(set(admin_dum.columns) | set(jvs_dum.columns))
    admin_dum = admin_dum.reindex(columns=all_cols, fill_value=0)
    jvs_dum = jvs_dum.reindex(columns=all_cols, fill_value=0)
    w_jvs = jvs['weight'].values
    smd_raw, smd_adj = {}, {}
    for col in all_cols:
        m_a = admin_dum[col].mean(); v_a = admin_dum[col].var()
        m_j = np.average(jvs_dum[col], weights=w_jvs)
        v_j = np.average((jvs_dum[col] - m_j)**2, weights=w_jvs)
        pool = np.sqrt((v_a + v_j) / 2)
        smd_raw[col] = (m_a - m_j) / pool if pool > 0 else 0
        m_a_w = np.average(admin_dum[col], weights=ipw_weights)
        v_a_w = np.average((admin_dum[col] - m_a_w)**2, weights=ipw_weights)
        pool_w = np.sqrt((v_a_w + v_j) / 2)
        smd_adj[col] = (m_a_w - m_j) / pool_w if pool_w > 0 else 0
    balance = pd.DataFrame({'Unadjusted': smd_raw, 'Adjusted': smd_adj})
    balance = balance.sort_values('Unadjusted', key=abs, ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(6, len(balance) * 0.22)))
    y_pos = np.arange(len(balance))
    ax.scatter(balance['Unadjusted'].abs(), y_pos, c='coral', s=40, zorder=3, label='Unadjusted')
    ax.scatter(balance['Adjusted'].abs(), y_pos, c='steelblue', s=40, zorder=3, label='Adjusted')
    ax.axvline(0.1, ls='--', color='black', lw=1, alpha=0.5)
    ax.set_yticks(y_pos); ax.set_yticklabels(balance.index, fontsize=6)
    ax.set_xlabel('|SMD|'); ax.set_title(f'Balans zmiennych {title}')
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout(); plt.show()

### Model 1: `~size`

In [ ]:
love_plot(admin, jvs, res1['ipw_weights'], title='(model 1: ~size)')

### Model 2: `~size+nace+region+private`

In [ ]:
love_plot(admin, jvs, res2['ipw_weights'], title='(model 2: ~size+nace+region+private)')